### Read the Current Eddy Tracks

In [ ]:
from pathlib import Path
from collections.abc import Callable
from typing import cast
import sys

import numpy as np
import pandas as pd

PROJECT_ROOT = Path('/Users/jerry/school/research/eddy-tracking')
sys.path.insert(0, str(PROJECT_ROOT))

from utils.config import METADATA_COLS, load_config
from eddy_tracking.packages.py_eddy_tracker.observations.tracking import (
    TrackEddiesObservations,
)
from eddy_tracking.packages.sdp.prediction import SDP_PIGMENT_COLUMNS

from matplotlib.axes import Axes
from matplotlib.figure import Figure


Constants / Configs

In [ ]:
EXPERIMENT = 'gulf_stream_20240305_20260531'
cfg = load_config(EXPERIMENT)
LON_RANGE = tuple(cfg['base']['region']['lon_range'])
LAT_RANGE = tuple(cfg['base']['region']['lat_range'])
DATA_DIR = PROJECT_ROOT / 'data' / EXPERIMENT
SILVER_DIR = DATA_DIR / 'silver'

ANTICYCLONE_TRACKED_DIR = SILVER_DIR / 'eddy_track' / 'anticyclone'
CYCLONE_TRACKED_DIR = SILVER_DIR / 'eddy_track' / 'cyclone'


The current experiment stores daily eddy tracks under `silver/eddy_track`.

The pipeline already fills permitted gaps and smooths the positions. Read these tracks directly so their IDs match the reflectance and pigment files.


### View Outputs of Eddy Tracking

In [ ]:
a = TrackEddiesObservations.load_file(
    str(ANTICYCLONE_TRACKED_DIR / 'anticyclone_tracks.zarr')
)
c = TrackEddiesObservations.load_file(
    str(CYCLONE_TRACKED_DIR / 'cyclone_tracks.zarr')
)


All tracked eddy trajectories after position smoothing (median half-window = 1, Loess half-window = 5). Line color encodes track duration; trajectories are wrapped to [-180, 180] longitude.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.collections import LineCollection, PolyCollection
import numpy as np

def plot_tracks(ax, tracked, color):
    for i, _, _ in tracked.iter_on('track'):
        lon = (tracked.longitude[i] + 180) % 360 - 180
        lat = tracked.latitude[i]
        points = np.column_stack((lon, lat)).reshape(-1, 1, 2) # (n_obs,), (n_obs,) -> (n_obs, 1, 2)
        segments = np.concatenate([points[:-1], points[1:]], axis=1) # (n_obs, 1, 2) -> (n_obs-1, 2, 2)
        n = len(segments)
        colors = np.zeros((n, 4))
        colors[:, :3] = color
        colors[:, 3] = np.linspace(0.15, 1.0, n)  # fade in over track lifetime
        lc = LineCollection(segments, colors=colors, linewidth=0.8)
        ax.add_collection(lc)

fig, (ax1, ax2) = cast(
    tuple[Figure, tuple[Axes, Axes]],
    plt.subplots(1, 2, figsize=(20, 6)),
)

plot_tracks(ax1, a, (0.8, 0.1, 0.1))
ax1.set_xlim(LON_RANGE)
ax1.set_ylim(LAT_RANGE)
ax1.set_aspect('equal')
ax1.set_title('Anticyclonic Tracks')
ax1.grid()

plot_tracks(ax2, c, (0.1, 0.2, 0.8))
ax2.set_xlim(LON_RANGE)
ax2.set_ylim(LAT_RANGE)
ax2.set_aspect('equal')
ax2.set_title('Cyclonic Tracks')
ax2.grid()

plt.tight_layout()
plt.show()

### Isolate Longest-Lived Eddies

Filter by lifetime and plot out their tracks. Pick out a few to obtain contours over time, and then obtain a lon/lat boolean mask for their contours.

Length refers to the number of observations, which is the count of times that a specific track id appears in the track array.

Time is days since 1950-01-01. Consider each individual eddy track to be non-overlapping subarrays of time. Within each subarray, time is strictly monotonically increasing.

In [ ]:
def count_calendar_days_per_track(tracked: TrackEddiesObservations) -> np.ndarray:
    """
    Compute the inclusive calendar duration (in days) of each track.

    Uses the contiguous ordering of the track array to find boundaries between tracks, then computes duration in days per track.

    Args:
        tracked: TrackEddiesObservations with contiguous track IDs.

    Returns:
        1D array of calendar days per track, indexed by track ID.
    """
    boundaries = np.flatnonzero(np.diff(tracked.track)) + 1
    boundaries = np.concatenate([[0], boundaries, [len(tracked.track)]]) # (n_tracks-1,) -> (n_tracks+1,)
    t_start = tracked.time[boundaries[:-1]]
    t_end = tracked.time[boundaries[1:] - 1]
    return t_end - t_start + 1

In [ ]:
a_cal_days = count_calendar_days_per_track(a)
c_cal_days = count_calendar_days_per_track(c)

print(
    f"polarity: anticyclonic\n"
    f"tracks: {len(a_cal_days)}\n"
    f"longest_days: {a_cal_days.max()}"
)
print(
    f"polarity: cyclonic\n"
    f"tracks: {len(c_cal_days)}\n"
    f"longest_days: {c_cal_days.max()}"
)

Find the longest-lived track IDs for each type.

In [ ]:
a_longest_ids = np.where(a_cal_days == a_cal_days.max())[0]
c_longest_ids = np.where(c_cal_days == c_cal_days.max())[0]

print(
    f"polarity: anticyclonic\n"
    f"longest_track_ids: {a_longest_ids}\n"
    f"longest_days: {a_cal_days.max()}"
)
print(
    f"polarity: cyclonic\n"
    f"longest_track_ids: {c_longest_ids}\n"
    f"longest_days: {c_cal_days.max()}"
)

Obtain new TrackEddiesObservations objects with only the longest lived eddies

In [ ]:
a_long = a.extract_ids(a_longest_ids)
c_long = c.extract_ids(c_longest_ids)

Trajectories of the longest-lived anticyclonic (left, red) and cyclonic (right, blue) eddies, colored by time progression. The pigment analysis selects the longest cyclone with both reflectance and pigment results.

In [ ]:
fig, (ax1, ax2) = cast(
    tuple[Figure, tuple[Axes, Axes]],
    plt.subplots(1, 2, figsize=(20, 7)),
)

for ax, tracked, title, cmap_name in [
    (ax1, a_long, 'Longest Anticyclonic Tracks', 'Reds'),
    (ax2, c_long, 'Longest Cyclonic Tracks', 'Blues'),
]:
    cmap = plt.get_cmap(cmap_name)
    unique_ids = np.unique(tracked.track)
    for track_id in unique_ids:
        eddy_mask = tracked.track == track_id
        lon = (tracked.longitude[eddy_mask] + 180) % 360 - 180
        lat = tracked.latitude[eddy_mask]

        # Fade segments from transparent to opaque over the track lifetime
        points = np.column_stack((lon, lat)).reshape(-1, 1, 2) # (n_obs,), (n_obs,) -> (n_obs, 1, 2)
        segments = np.concatenate([points[:-1], points[1:]], axis=1) # (n_obs, 1, 2) -> (n_obs-1, 2, 2)
        n = len(segments)
        seg_colors = np.zeros((n, 4))
        seg_colors[:, :3] = cmap(0.7)[:3]
        seg_colors[:, 3] = np.linspace(0.15, 1.0, n)
        lc = LineCollection(segments, colors=seg_colors, linewidth=1.5)
        ax.add_collection(lc)

        ax.plot(lon[0], lat[0], 'o', color=cmap(0.7), markersize=6)
        ax.plot(lon[-1], lat[-1], 's', color=cmap(0.9), markersize=6)

    ax.set_xlim(LON_RANGE)
    ax.set_ylim(LAT_RANGE)
    ax.set_aspect('equal')
    ax.set_title(title)
    ax.set_xlabel('Longitude')
    ax.set_ylabel('Latitude')
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('longest_lived_tracks.png', dpi=150, bbox_inches='tight')
plt.show()

Speed contour evolution of the longest-lived eddies over their lifetimes. The speed contour follows the boundary with the greatest mean speed around the eddy.

In [ ]:
def get_id_indices(tracked: TrackEddiesObservations, track_id: int) -> np.ndarray:
    """Obtain the row indices belonging to a specific track (e.g., for speed_contour_lon/lat)."""
    return np.where(tracked.track == track_id)[0]

a_longest_idxs = {} # map track id to row indices
c_longest_idxs = {}

for id in a_longest_ids:
    a_longest_idxs[id] = get_id_indices(a_long, id)
for id in c_longest_ids:
    c_longest_idxs[id] = get_id_indices(c_long, id)

# Plot contour evolution, subsampled weekly
SUBSAMPLE = 7

fig, (ax1, ax2) = cast(
    tuple[Figure, tuple[Axes, Axes]],
    plt.subplots(1, 2, figsize=(20, 7)),
)

for ax, tracked, idxs_dict, title, cmap_name in [
    (ax1, a_long, a_longest_idxs, 'Longest Anticyclonic Contours (weekly)', 'Reds'),
    (ax2, c_long, c_longest_idxs, 'Longest Cyclonic Contours (weekly)', 'Blues'),
]:
    cmap = plt.get_cmap(cmap_name)
    for track_id, rows in idxs_dict.items():
        sampled = rows[::SUBSAMPLE]
        n_sampled = len(sampled)
        for j, i in enumerate(sampled):
            lon = (tracked.contour_lon_s[i] + 180) % 360 - 180
            lat = tracked.contour_lat_s[i]

            # Close the polygon
            lon = np.append(lon, lon[0])
            lat = np.append(lat, lat[0])

            color = cmap(0.3 + 0.6 * j / max(n_sampled - 1, 1))
            ax.plot(lon, lat, color=color, linewidth=0.8, alpha=0.7)

            # Plot eddy center
            center_lon = (tracked.longitude[i] + 180) % 360 - 180
            center_lat = tracked.latitude[i]
            ax.plot(center_lon, center_lat, '.', color=color, markersize=4)

        # Label at first contour center
        first_i = rows[0]
        label_lon = (tracked.longitude[first_i] + 180) % 360 - 180
        label_lat = tracked.latitude[first_i]
        ax.annotate(
            str(track_id), (label_lon, label_lat),
            fontsize=9, fontweight='bold', color=cmap(0.9),
            textcoords='offset points', xytext=(5, 5),
            bbox=dict(boxstyle='round,pad=0.2', fc='white', ec='none', alpha=0.7),
        )

    ax.set_xlim(LON_RANGE)
    ax.set_ylim(LAT_RANGE)
    ax.set_aspect('equal')
    ax.set_title(title)
    ax.set_xlabel('Longitude')
    ax.set_ylabel('Latitude')
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

Weekly speed contours for the longest cyclonic tracks in `silver/eddy_track/cyclone`. The plot labels the five longest tracks and displays ten tracks.


In [ ]:
# Weekly contour evolution for a readable subset of long-lived cyclonic eddies
N_PLOT = 10
N_LABEL = 5
SUBSAMPLE = 7

# Rank cyclone track IDs by inclusive calendar duration, longest first
c_ranked_ids = sorted(range(len(c_cal_days)), key=lambda tid: (-c_cal_days[tid], tid))
cyclone_plot_ids = c_ranked_ids[:N_PLOT]
cyclone_label_ids = set(cyclone_plot_ids[:N_LABEL])
cyclone_plot_rows = {track_id: get_id_indices(c, track_id) for track_id in cyclone_plot_ids}

fig, ax = cast(
    tuple[Figure, Axes],
    plt.subplots(figsize=(12, 7)),
)
cmap = plt.get_cmap('Blues')

for rank, track_id in enumerate(cyclone_plot_ids):
    rows = cyclone_plot_rows[track_id]
    sampled = rows[::SUBSAMPLE]
    n_sampled = len(sampled)
    is_labeled = track_id in cyclone_label_ids

    for j, i in enumerate(sampled):
        lon = (c.contour_lon_s[i] + 180) % 360 - 180
        lat = c.contour_lat_s[i]

        lon = np.append(lon, lon[0])
        lat = np.append(lat, lat[0])

        color_level = 0.22 + 0.60 * j / max(n_sampled - 1, 1)
        color = cmap(color_level)
        linewidth = 1.05 if is_labeled else 0.7
        alpha = 0.82 if is_labeled else 0.38
        ax.plot(lon, lat, color=color, linewidth=linewidth, alpha=alpha, zorder=2)

        center_lon = (c.longitude[i] + 180) % 360 - 180
        center_lat = c.latitude[i]
        ax.plot(
            center_lon, center_lat, '.', color=color,
            markersize=4 if is_labeled else 2.5,
            alpha=0.9 if is_labeled else 0.55,
            zorder=3,
        )

    if is_labeled:
        label_i = sampled[len(sampled) // 2]
        label_lon = (c.longitude[label_i] + 180) % 360 - 180
        label_lat = c.latitude[label_i]
        ax.annotate(
            str(track_id), (label_lon, label_lat),
            fontsize=10,
            fontweight='bold',
            color=cmap(0.9),
            textcoords='offset points',
            xytext=(6, 6),
            bbox=dict(boxstyle='round,pad=0.25', fc='white', ec='none', alpha=0.85),
            zorder=4,
        )

ax.set_xlim(LON_RANGE)
ax.set_ylim(LAT_RANGE)
ax.set_aspect('equal')
ax.set_title(f'Longest Cyclonic Contours (weekly, top {N_PLOT})')
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
ax.grid(alpha=0.3)

out_fp = Path('visuals') / 'cyclonic_contours_weekly_subset.png'
out_fp.parent.mkdir(parents=True, exist_ok=True)
plt.tight_layout()
plt.savefig(out_fp, dpi=150, bbox_inches='tight')
plt.show()

print(f'cyclone_track_ids_plotted: {cyclone_plot_ids}')
print(f'cyclone_track_ids_labeled: {sorted(cyclone_label_ids)}')
print(
    f'figure_path: {out_fp}\n'
    f'status: saved'
)



### Read PACE Reflectance Data

Read the current experiment's per-eddy Parquet files from `silver/collocate_pace` and `silver/pigments`.

The PACE data use eight-day composites. Each file labels its pixels with the composite midpoint date. The selected contour can come from another day within that composite.

`coverage` is the fraction of pixels inside the speed contour with a finite reflectance spectrum. The experiment requires at least 80% coverage.

Pigment files can contain fewer pixels than reflectance files. Use each pigment row's own date and coordinates; do not copy coordinates by row position.


In [ ]:
rrs_paths = {
    int(file.stem.split('_')[1]): file
    for file in (SILVER_DIR / 'collocate_pace' / 'cyclone').glob('eddy_*_rrs.parquet')
}
pigment_paths = {
    int(file.stem.split('_')[1]): file
    for file in (SILVER_DIR / 'pigments' / 'cyclone').glob('eddy_*_pigments.parquet')
}
available_ids = set(rrs_paths) & set(pigment_paths) & set(c_ranked_ids)
if not available_ids:
    raise ValueError(f'No cyclone has both reflectance and pigment data in {EXPERIMENT}.')

c_eddy_id = next(track_id for track_id in c_ranked_ids if track_id in available_ids)
print(f'experiment: {EXPERIMENT}')
print(f'cyclone_track_id: {c_eddy_id}')
print(f'lifetime_days: {c_cal_days[c_eddy_id]:.0f}')


In [ ]:
rrs_fp = rrs_paths[c_eddy_id]
inputs_df = pd.read_parquet(rrs_fp)
rrs_cols = [column for column in inputs_df if column.startswith('Rrs_')]
wavelengths = np.array([float(column.split('_')[1]) for column in rrs_cols])

print(f'reflectance_path: {rrs_fp}')
print(f'reflectance_pixels: {len(inputs_df)}')
print(f'composite_dates: {inputs_df["date"].nunique()}')
print(f'wavelengths: {len(wavelengths)}')


The pipeline prepares spectra on a 1 nm grid and applies a 5 nm mean before the pigment model uses the 400-700 nm interval.

It also samples temperature and salinity for each pixel. This notebook reads the saved pigment results without a new model run.


### Read Pigment Results

In [ ]:
pigments_fp = pigment_paths[c_eddy_id]
pigments_df = pd.read_parquet(pigments_fp)
pigment_cols = list(SDP_PIGMENT_COLUMNS)

print(f'pigment_path: {pigments_fp}')
print(f'pigment_pixels: {len(pigments_df)}')
print(f'composite_dates: {pigments_df["date"].nunique()}')
print(pigments_df[pigment_cols].describe())


### Check whether the values predicted from SDP model make sense.

First check whether the relationship between predicted values makes sense.

Use DPA framework to estimate the contribution of each major phytoplankton group to total chlorophyll a, using significant accessory pigments with empiricle weights.

In [ ]:
# Uitz diagnostic pigment analysis
# TChla = 1.41*Fuco + 1.41*Perid + 1.27*HexFuco + 0.35*ButFuco + 0.60*Allo  + 1.01*TChlb + 0.86*Zea

DPA_WEIGHTS = {
    'Fuco':    1.41,
    'Perid':   1.41,
    'HexFuco': 1.27,
    'ButFuco': 0.35,
    'Allo':    0.60,
    'MV chlb': 1.01, # use in place of TChlb
    'Zea':     0.86,
}

tchla_pred = pigments_df[list(DPA_WEIGHTS)].mul(pd.Series(DPA_WEIGHTS)).sum(axis=1)
diffs = tchla_pred - pigments_df['T chla']

# Build a per-composite comparison
rows = []
dates = pigments_df['date'].dt.date
unique_dates = sorted(dates.unique())

for d in unique_dates:
    mask = (dates == d)

    row = {
        'date': d,
        'n_pixels': mask.sum(),
        'mean_chla_sdp': pigments_df.loc[mask, 'T chla'].mean(),
        'mean_chla_dpa': tchla_pred[mask].mean(),
        'median_chla_sdp': pigments_df.loc[mask, 'T chla'].median(),
        'median_chla_dpa': tchla_pred[mask].median()
    }
    rows.append(row)

dpa_df = pd.DataFrame(rows)
print(dpa_df)

fig, (ax1, ax2) = cast(
    tuple[Figure, tuple[Axes, Axes]],
    plt.subplots(1, 2, figsize=(14, 6)),
)

for ax, sdp_col, dpa_col, stat in [
    (ax1, 'mean_chla_sdp', 'mean_chla_dpa', 'Mean'),
    (ax2, 'median_chla_sdp', 'median_chla_dpa', 'Median'),
]:
    ax.scatter(dpa_df[sdp_col], dpa_df[dpa_col], s=50, zorder=3)

    lim = max(dpa_df[sdp_col].max(), dpa_df[dpa_col].max()) * 1.15
    ax.plot([0, lim], [0, lim], 'k--', linewidth=1, label='1:1 line')

    ax.set(xlabel=f'{stat} TChla, SDP (mg/m³)',
        ylabel=f'{stat} TChla, DPA (mg/m³)',
        title=f'Composite {stat}: SDP vs DPA',
        xlim=(0, lim), ylim=(0, lim))
    ax.set_aspect('equal')
    ax.legend()
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

Check the fraction of zeros after running the model, since negative values are clipped to zero.

In [ ]:
# Negative SDP predictions are clipped to 0, so a high zero fraction signals frequent unphysical (negative) model outputs for that pigment.

pigment_cols = list(SDP_PIGMENT_COLUMNS)
zero_frac = (pigments_df[pigment_cols] == 0).mean().sort_values(ascending=False)

fig, ax = cast(
    tuple[Figure, Axes],
    plt.subplots(figsize=(12, 5)),
)
bars = ax.bar(range(len(zero_frac)), zero_frac.values, color='steelblue')
ax.xaxis.set_ticks(range(len(zero_frac)))
ax.xaxis.set_ticklabels(zero_frac.index, rotation=45, ha='right')
ax.set(ylabel='Fraction of pixels = 0', title='Zero Fraction per Pigment', ylim=(0, min(1.05, float(zero_frac.max()) + 0.12)))
ax.grid(alpha=0.3, axis='y')

for i, (name, frac) in enumerate(zero_frac.items()):
    ax.text(i, frac + 0.01, f'{frac:.1%}', ha='center', va='bottom', fontsize=8)

plt.tight_layout()
plt.show()


Check pigment ratio to total chlorophyll a and whether that matches with literature.

In [ ]:
# Pigment-to-TChla ratios vs. literature ranges. 
#
# Sources:
#   [1] Kramer & Siegel (2019) https://pmc.ncbi.nlm.nih.gov/articles/PMC7043335/
#   [2] Kramer et al. (2020) https://doi.org/10.3389/fmars.2020.00215
#   [3] Goericke & Repeta (1992) DVchla in subtropical N. Atlantic
#   [4] Higgins et al. (2011) Phytoplankton Pigments, Cambridge Univ. Press
#   [5] Aiken et al. (2009) https://doi.org/10.1016/j.dsr2.2008.09.017

LITERATURE = {
    'Perid':     {'range': (0.01, 0.05), 'group': 'Dinoflagellates'}, # regional [1]: BOUS 0.019, CAR 0.023; global 0.017
    'Fuco':      {'range': (0.05, 0.30), 'group': 'Diatoms'}, # regional [1]: BOUS 0.056, CAR 0.065; global 0.140
    'Zea':       {'range': (0.05, 0.30), 'group': 'Cyanobacteria'}, # regional [1]: BOUS 0.134, CAR 0.294
    'DV chla':   {'range': (0.10, 0.40), 'group': 'Prochlorococcus'}, # regional [3]: up to 40% of TChla, subtropical N. Atl
    'chl c3':    {'range': (0.02, 0.15), 'group': 'Haptophytes'}, # global [4,5]: CHEMTAX ratio matrices
    'HexFuco':   {'range': (0.10, 0.40), 'group': 'Haptophytes'}, # regional [1]: BOUS 0.253, CAR 0.102
    'MV chlb':   {'range': (0.00, 0.15), 'group': 'Green algae'}, # regional [1]: BOUS 0.000, CAR 0.046; global 0.045
}

valid = pigments_df['T chla'] > 0
ratios = {}
for pig in LITERATURE:
    ratios[pig] = pigments_df.loc[valid, pig] / pigments_df.loc[valid, 'T chla']

print(f"{'Pigment':>10s}  {'Group':<18s}  {'Model Median':>13s}  {'Model IQR':>16s}  "
    f"{'Range from Lit.':>16s}")
print('-' * 80)

for pig, info in LITERATURE.items():
    r = ratios[pig]
    med = r.median()
    q25, q75 = r.quantile(0.25), r.quantile(0.75)
    lo, hi = info['range']

    print(f'{pig:>10s}  {info["group"]:<18s}  {med:13.3f}  [{q25:.3f}, {q75:.3f}]  '
        f'[{lo:.3f}, {hi:.2f}]')

fig, ax = cast(
    tuple[Figure, Axes],
    plt.subplots(figsize=(12, 5)),
)
names = list(LITERATURE.keys())
x = np.arange(len(names))

lows = [LITERATURE[p]['range'][0] for p in names]
highs = [LITERATURE[p]['range'][1] for p in names]
ax.bar(x, [h - l for h, l in zip(highs, lows)], bottom=lows,
    width=0.6, color='lightgreen', alpha=0.5, label='Range from literature')

medians = [ratios[p].median() for p in names]
q25s = [ratios[p].quantile(0.25) for p in names]
q75s = [ratios[p].quantile(0.75) for p in names]
# yerr rows are the lower then the upper offset from the median, the two-row layout errorbar expects
ax.errorbar(x, medians, yerr=[[m - q for m, q in zip(medians, q25s)],
        [q - m for m, q in zip(medians, q75s)]],
    fmt='o', color='black', capsize=5, markersize=7, label='Model median (IQR)')

ax.xaxis.set_ticks(x)
ax.xaxis.set_ticklabels([f'{n}\n({LITERATURE[n]["group"]})' for n in names], fontsize=9)
ax.set(ylabel='Pigment / TChla ratio', title='Observed Pigment Ratios vs. Literature Ranges')
ax.legend()
ax.grid(alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

View how numerical values of pigments change over time.

In [ ]:
_pig_cols = [
    column for column in SDP_PIGMENT_COLUMNS if column != "T chla"
]

_dates = pd.to_datetime(pigments_df["date"])
_grouped = pigments_df.groupby(_dates.dt.date)
_daily_mean = cast(pd.DataFrame, _grouped[_pig_cols].mean())
_daily_std = cast(pd.DataFrame, _grouped[_pig_cols].std())
_daily_count = _grouped[_pig_cols[0]].count()

_PFT_MAP = {
    "Perid":   "Dinoflagellates",
    "Fuco":    "Diatoms",
    "Zea":     "Cyanobacteria",
    "DV chla": "Cyanobacteria",
    "chl c3":  "Haptophytes",
    "HexFuco": "Haptophytes",
    "MV chlb": "Green algae",
}

fig, axes = plt.subplots(3, 4, figsize=(18, 10), constrained_layout=True)
fig.suptitle(
    f"Eddy #{c_eddy_id} - mean pigment concentration over time",
    fontsize=16, fontweight="bold",
)

for i, pig in enumerate(_pig_cols):
    ax = cast(Axes, axes.flat[i])
    x = _daily_mean.index
    y = _daily_mean[pig].to_numpy()
    err = _daily_std[pig].to_numpy()

    cast(Callable[..., PolyCollection], ax.fill_between)(
        x, y - err, y + err, alpha=0.25, color="steelblue"
    )
    ax.plot(x, y, "o-", color="steelblue", markersize=4, linewidth=1.5)

    # Label with PFT if diagnostic
    label = pig
    if pig in _PFT_MAP:
        label += f" \u2192 {_PFT_MAP[pig]}"
    ax.set_title(label, fontsize=9)

    ax.set_ylabel("mg/m\u00b3", fontsize=8)
    ax.tick_params(labelsize=7)
    ax.tick_params(axis="x", rotation=45)
    ax.grid(alpha=0.3)

    # Annotate pixel count on top of every fourth point
    for xi, yi, n in zip(x[::4], y[::4], _daily_count.to_numpy()[::4]):
        ax.annotate(f"{n:,}", (xi, yi), textcoords="offset points",
            xytext=(0, 8), fontsize=6, ha="center", color="gray")

plt.show()

print(f"\n{'Date':<14s}  {'n_pixels':>8s}", end="")
for pig in _pig_cols:
    print(f"  {pig:>10s}", end="")
print()
print("-" * (24 + 12 * len(_pig_cols)))
for date in _daily_mean.index:
    n = _daily_count.loc[date]
    print(f"{str(date):<14s}  {n:>8,d}", end="")
    for pig in _pig_cols:
        print(f"  {_daily_mean.loc[date, pig]:>10.4f}", end="")
    print()

In [ ]:
_pct_levels = [5, 25, 50, 75, 95]

# Compute percentiles per date for each pigment
_daily_pcts = {}  # pigment -> {pct_level: array over dates}
for pig in _pig_cols:
    _daily_pcts[pig] = {}
    for p in _pct_levels:
        _daily_pcts[pig][p] = _grouped[pig].quantile(p / 100.0).values

_pct_dates = _daily_mean.index  # same date index as the mean cell

fig, axes = plt.subplots(3, 4, figsize=(18, 10), constrained_layout=True)
fig.suptitle(
    f"Eddy #{c_eddy_id} - pigment concentration percentiles over time",
    fontsize=16, fontweight="bold",
)

for i, pig in enumerate(_pig_cols):
    ax = cast(Axes, axes.flat[i])

    ax.fill_between(
        _pct_dates,
        _daily_pcts[pig][5],
        _daily_pcts[pig][95],
        alpha=0.15, color="steelblue", label="p5–p95",
    )
    ax.fill_between(
        _pct_dates,
        _daily_pcts[pig][25],
        _daily_pcts[pig][75],
        alpha=0.30, color="steelblue", label="p25–p75",
    )
    ax.plot(
        _pct_dates,
        _daily_pcts[pig][50],
        "o-", color="steelblue", markersize=3, linewidth=1.5, label="median",
    )

    # Label with PFT if diagnostic
    label = pig
    if pig in _PFT_MAP:
        label += f" \u2192 {_PFT_MAP[pig]}"
    ax.set_title(label, fontsize=9)

    ax.set_ylabel("mg/m\u00b3", fontsize=8)
    ax.tick_params(labelsize=7)
    ax.tick_params(axis="x", rotation=45)
    ax.grid(alpha=0.3)

    # Add legend only on first subplot to avoid clutter
    if i == 0:
        ax.legend(fontsize=7, loc="upper right")

# The 12 pigments fill the 3x4 grid exactly today, so this loop hides nothing
for j in range(len(_pig_cols), len(axes.flat)):
    axes.flat[j].set_visible(False)

plt.show()

View relative concentrations of various pigments within eddy over time.

In [ ]:
from datetime import datetime as _dt

from collocate_pace import build_date_eddy_index, collect_eddies_for_window
from eddy_tracking.utils.subset import parse_date_range

# Coverage from the current reflectance table (for title annotation only)
_coverage_per_date = inputs_df.groupby(
    inputs_df["date"].dt.strftime("%Y-%m-%d")
)["coverage"].first()

date_index = build_date_eddy_index(
    c,
    'cyclone',
    track_ids={c_eddy_id},
    region=cfg['collocate_pace'].get('region'),
    date_range=parse_date_range(cfg['collocate_pace'].get('date_range')),
)
pace_dir = DATA_DIR / 'bronze' / cfg['base']['data']['pace_dir']
contours_by_date = {}
for pace_path in sorted(pace_dir.glob('PACE_OCI.*_*.L3m.8D.*.nc')):
    start_text, end_text = pace_path.name.split('.')[1].split('_')
    window_start = _dt.strptime(start_text, '%Y%m%d').date()
    window_end = _dt.strptime(end_text, '%Y%m%d').date()
    observations = collect_eddies_for_window(date_index, window_start, window_end)
    if observations:
        midpoint = window_start + (window_end - window_start) / 2
        contours_by_date[midpoint.strftime('%Y-%m-%d')] = observations[0]

# All dates present in the (already-filtered) pigments data - no additional coverage threshold
SNAPSHOT_DATES = sorted(pigments_df["date"].dt.strftime("%Y-%m-%d").unique())

print(
    f"eddy_id: {c_eddy_id}\n"
    f"snapshot_dates: {len(SNAPSHOT_DATES)}"
)
for d in SNAPSHOT_DATES:
    day_n = (pigments_df["date"].dt.strftime("%Y-%m-%d") == d).sum()
    coverage = _coverage_per_date.get(d, float("nan"))
    print(
        f"date: {d}\n"
        f"pixels: {day_n:,}\n"
        f"coverage_percent: {coverage:.1%}"
    )

# 12 accessory pigment columns (exclude T chla)
PIGMENT_COLS = [
    column for column in SDP_PIGMENT_COLUMNS
    if column != "T chla"
]

# Key diagnostic pigment → PFT mapping (from clustering analysis)
PFT_MAP = {
    "Perid":   "Dinoflagellates",
    "Fuco":    "Diatoms",
    "Zea":     "Cyanobacteria",
    "DV chla": "Cyanobacteria",
    "chl c3":  "Haptophytes",
    "HexFuco": "Haptophytes",
    "MV chlb": "Green algae",
}

# Percentile bounds for robust normalisation
_PCT_LO, _PCT_HI = 5, 95

for date_str in SNAPSHOT_DATES:
    day_pig = cast(
        pd.DataFrame,
        pigments_df[pigments_df["date"].dt.strftime("%Y-%m-%d") == date_str],
    )
    n_pixels = len(day_pig)
    coverage = _coverage_per_date.get(date_str, float("nan"))

    observation = contours_by_date[date_str]
    clon = observation.contour_lon
    clat = observation.contour_lat
    clon_closed = np.append(clon, clon[0])
    clat_closed = np.append(clat, clat[0])

    # Reconstruct the regular grid from the sparse valid points
    unique_lon = np.unique(day_pig["pixel_lon"].to_numpy())
    unique_lat = np.unique(day_pig["pixel_lat"].to_numpy())

    # Grid spacing (L3 is uniform 4km ≈ 0.0417°)
    dlon = np.median(np.diff(unique_lon))
    dlat = np.median(np.diff(unique_lat))

    # Build a lookup from (lon, lat) → row index for fast 2D placement
    lon_to_j = {v: j for j, v in enumerate(unique_lon)}
    lat_to_i = {v: i for i, v in enumerate(unique_lat)}

    # pcolormesh cell edges: N+1 edges for N cell centers
    lon_edges = np.concatenate([unique_lon - dlon / 2, [unique_lon[-1] + dlon / 2]])
    lat_edges = np.concatenate([unique_lat - dlat / 2, [unique_lat[-1] + dlat / 2]])

    fig, axes = plt.subplots(3, 4, figsize=(18, 13), constrained_layout=True)
    fig.suptitle(
        f"Eddy #{c_eddy_id} pigment concentrations - composite midpoint {date_str}\n"
        f"{n_pixels:,} pixels, coverage={coverage:.1%}",
        fontsize=16,
        fontweight="bold",
    )

    axes_flat = axes.flatten()

    for i, pigment in enumerate(PIGMENT_COLS):
        ax = cast(Axes, axes_flat[i])
        vals = day_pig[pigment].to_numpy()

        # Percentile-based normalisation: clip then scale to [0, 1]
        p5 = np.percentile(vals, _PCT_LO)
        p95 = np.percentile(vals, _PCT_HI)
        if p95 - p5 > 0:
            clipped = np.clip(vals, p5, p95)
            norm_vals = (clipped - p5) / (p95 - p5)
        else:
            norm_vals = np.full_like(vals, 0.5)

        # Place normalised values into a 2D grid (NaN where no valid pixel)
        grid = np.full((len(unique_lat), len(unique_lon)), np.nan)
        for nv, lon_val, lat_val in zip(
            norm_vals, day_pig["pixel_lon"].to_numpy(), day_pig["pixel_lat"].to_numpy()
        ):
            grid[lat_to_i[lat_val], lon_to_j[lon_val]] = nv

        pc = ax.pcolormesh(
            lon_edges, lat_edges, grid,
            cmap="RdYlBu_r",
            vmin=0,
            vmax=1,
            rasterized=True,
        )

        ax.plot(clon_closed, clat_closed, color="black", linewidth=1.5)

        label = pigment
        if pigment in PFT_MAP:
            label += f" → {PFT_MAP[pigment]}"
        ax.set_title(f"{label}\n[p5={p5:.4f}  p95={p95:.4f} mg/m³]", fontsize=9)
        ax.set_aspect("equal")
        ax.tick_params(labelsize=7)

        if i % 4 == 0:
            ax.set_ylabel("Latitude", fontsize=8)
        if i >= 8:
            ax.set_xlabel("Longitude", fontsize=8)

    cbar = fig.colorbar(
        plt.cm.ScalarMappable(norm=plt.Normalize(0, 1), cmap="RdYlBu_r"),
        ax=axes, location="right", shrink=0.8, pad=0.02,
    )
    cbar.set_label("Normalised concentration (0 = p5, 1 = p95)", fontsize=10)

    plt.show()

### Translate phytoplankton pigments to phytoplankton groups

Two primary approaches:
- CHEMTAX
    - Assumes ratio of a specific pigment to total chlorophyll a for a given phytoplankton group is fixed/known.
    - Assumes linear independence between pigments, but multicolinearity could mess up the matrix inversion process.
- Hierarchical clustering + empirical orthogonal functions
    - https://pmc.ncbi.nlm.nih.gov/articles/PMC7043335/
    

For any 2 branches connected by a horizontal link, the height of the horizontal segment on the y-axis is the distance. The lower two are, the more similar.

In [ ]:
import scipy.cluster.hierarchy as sch
from scipy.spatial.distance import squareform

keep_cols = [column for column in SDP_PIGMENT_COLUMNS if column != 'T chla']
valid = pigments_df['T chla'] > 0

ratios = pigments_df.loc[valid, keep_cols].div(pigments_df.loc[valid, 'T chla'], axis=0) # 12 cols

corr = ratios.corr() # 12x12 symmetric matrix
dist = 1 - corr
condensed = squareform(dist, checks=False)
linkage = sch.linkage(condensed, method='ward')

clusters = sch.fcluster(linkage, t=0.65, criterion='distance')

fig, ax = cast(
    tuple[Figure, Axes],
    plt.subplots(figsize=(10, 6)),
)
sch.dendrogram(
    linkage, labels=keep_cols, ax=ax, color_threshold=0.65,
    leaf_rotation=45, leaf_font_size=9,
)
plt.tight_layout()
plt.show()



- **Diatoms**: Fuco + chl c1+c2
- **Haptophytes**: HexFuco + ButFuco + chl c3
- **Cyanobacteria**: Zea + DV chla
- **Dinoflagellates**: Perid
- **Green algae**: MV chlb + Viola + Neo
- **Cryptophytes**: Allo

Kramer 2019 also found that allo (cryptophytes) clusters with green algae (right side).
Not sure about HexFuco and ButFuco being with Perid.

Groups that should be abundant near the gulf stream:
- Haptophytes
- Cyanobacteria
- Diatoms, but only when there is an abundance of nutrients
- Cryptophytes should be least abundant

**Next steps**

Quality Verification:
- Some days have partial QC flags; while the sensors may not have flagged other nearby pixels as having issues, this model seems fairly sensitive to Rrs changes.
- Look for days where TChla predictions are exactly 0, and where pixels have negative Rrs values
- Look for the peaks and lows in the mean/percentile plots

Moving from pigment clusters / PFT groups to a quantifiable metric. Like a percentage composition, or some raw number indicating abundance given pigment concentrations.
- What existing methods are there?
- What factors influence this translation? Season? Lighting conditions? Geographic region?

Also curious about the methods involved from the Rrs -> pigment step and pigment -> phytoplankton step:
- Using tchla as a predictor, since pigment concentrations clearly vary depending on environment (oligotrophic vs. nutrient-rich coastal waters). Open ocean is fine, but in coastal waters where results are much more impactful research is doing worse relatively.